# Real-data validation, Part 1: pure-compound literature viscosity curves

Reproduces the code behind paper Sections 3.2, 3.3 and 3.4 (Tg-predictor validation, full-pipeline validation
against 53 real literature viscosity curves, the dilute-vs-glassy regime split, and the follow-up glassy-regime
diagnosis: pure-component-viscosity vs. mixing-rule decomposition, fragility-parameter sensitivity, and the
ideal-mixing-rule ablation on the deep-glassy sugars).

**Pipeline under test:** SMILES -> ML-predicted Tg (Armeli/Peters/Koop 2023) -> VTF pure-component viscosity ->
AIOMFAC composition -> G2020 (Gervasi/Topping/Zuend 2020) mixture-viscosity mixing rule -> predicted log10(eta) at
a given water mass fraction, compared against literature-measured values.

**Data:** 53 (compound, source) water-activity/viscosity curves for 16 atmospherically relevant organics, from
the Supplement of Gervasi, Topping and Zuend (2020) ("Aqueous Binary Systems"), plus the BIMOG database's measured
Tg values (`bimog_unified.json`) for 11 of those 16 compounds.

## 0. Setup

In [1]:
import os, sys
if "google.colab" in sys.modules:   # Colab: install the package; locally: use the installed aiomfac_py
    if not os.path.exists("aiomfac-python"):
        !git clone -q https://github.com/SeoulTechPSE/aiomfac-python.git
    %cd aiomfac-python
    !python setup_aiomfac.py --smiles --no-editable -q 2>&1 | tail -20
    !pip install -q -e ".[tgml,tgml-smiles]" 2>&1 | tail -10
    !pip install -q scipy matplotlib
    %cd ..
import warnings
from sklearn.exceptions import InconsistentVersionWarning
warnings.filterwarnings("ignore", category=InconsistentVersionWarning)
import aiomfac_py
print("setup done; aiomfac_py", aiomfac_py.__version__, "from", os.path.dirname(aiomfac_py.__file__))

setup done; aiomfac_py 1.3.0 from /Users/kyyoo/Downloads/aiomfac_/aiomfac-python/src/aiomfac_py


In [2]:
# Upload bimog_unified.json (Colab only) -- the real-compound table with literature-measured Tg values.
try:
    from google.colab import files
    if not os.path.exists('bimog_unified.json'):
        print("Select bimog_unified.json to upload...")
        uploaded = files.upload()
except ImportError:
    print("Not running in Colab -- make sure bimog_unified.json is in the current directory.")

Not running in Colab -- make sure bimog_unified.json is in the current directory.


In [3]:
# Upload gervasi_supp.zip (Colab only) -- the Gervasi, Topping & Zuend (2020) Supplement data
# ("Aqueous Binary Systems" + "Aqueous SOA Systems" folders), then unzip it into the working directory.
try:
    from google.colab import files
    if not os.path.exists("Aqueous Binary Systems"):
        print("Select gervasi_supp.zip to upload...")
        uploaded = files.upload()
        import zipfile
        with zipfile.ZipFile("gervasi_supp.zip") as zf:
            zf.extractall(".")
except ImportError:
    print("Not running in Colab -- make sure the 'Aqueous Binary Systems' and 'Aqueous SOA Systems' "
          "folders are in the current directory (unzip gervasi_supp.zip here).")
import os
print(os.listdir("."))

Not running in Colab -- make sure the 'Aqueous Binary Systems' and 'Aqueous SOA Systems' folders are in the current directory (unzip gervasi_supp.zip here).
['gervasi_supp.zip', 'aiomfac_validation_01_pure_compounds.ipynb', 'aiomfac_validation_02_soa_proxies.ipynb', 'bimog_unified.json', 'log_01_pure_compounds.txt', 'Aqueous SOA Systems', 'Aqueous Binary Systems']


## 1. Parse the literature viscosity curves (paper Section 2.4)

Parses the 57 "Aqueous Binary Systems" CSVs into one unified table of (compound, source) curves. 11 of the 16
compounds reuse `bimog_unified.json`'s SMILES and measured Tg; 5 compounds not in BIMOG get a hand-supplied
SMILES (no measured Tg available for those).

In [4]:
import csv, glob, json, os, re
import sys
sys.path.insert(0, "aiomfac-python/src")

DATA_DIR = "Aqueous Binary Systems"
OUT = "literature_curves.json"

EXTRA_SMILES = {
    "acetic_acid": "CC(=O)O",
    "1,4-butanediol": "OCCCCO",
    "glutaric_acid": "OC(=O)CCCC(=O)O",
    "maleic_acid": "OC(=O)C=CC(=O)O",
    "maltose": "OC1OC(CO)C(O)C(O)C1OC1OC(CO)C(O)C(O)C1O",
}
BIMOG_NAME_MAP = {
    "1,2,4-butanetriol": "1,2,4-butanetriol", "1,2,6-hexanetriol": "1,2,6-hexanetriol",
    "citric_acid": "citric acid", "erythritol": "erythritol", "fructose": "fructose",
    "glucose": "glucose", "glycerol": "glycerol", "raffinose": "raffinose",
    "sorbitol": "sorbitol", "sucrose": "sucrose", "trehalose": "trehalose",
}

with open("bimog_unified.json") as f:
    bimog = json.load(f)
bimog_by_name = {d["name"]: d for d in bimog}

curves = []
n_rows_total = 0
for path in sorted(glob.glob(os.path.join(DATA_DIR, "*.csv"))):
    fname = os.path.basename(path)
    stem = fname.split("+water_viscosity_")[0]
    source_part = fname.split("+water_viscosity_")[1].rsplit(".csv", 1)[0]
    m = re.search(r"@(\d+)K", source_part)
    T_from_name = float(m.group(1)) if m else None
    source = re.sub(r"@\d+K.*", "", source_part)

    if stem in BIMOG_NAME_MAP:
        bimog_entry = bimog_by_name[BIMOG_NAME_MAP[stem]]
        smiles = bimog_entry["smiles"]
        tg_measured_K = bimog_entry["tg_K"]
    elif stem in EXTRA_SMILES:
        smiles = EXTRA_SMILES[stem]
        tg_measured_K = None
    else:
        print(f"SKIP (no SMILES mapping): {stem}")
        continue

    with open(path, newline="", encoding="utf-8-sig") as f:
        reader = csv.reader(f)
        rows = list(reader)
    header = [h.strip() for h in rows[1]]
    idx = {h: i for i, h in enumerate(header)}

    def col(row, key, default=""):
        i = idx.get(key)
        if i is None or i >= len(row):
            return default
        return row[i].strip()

    pts = []
    for row in rows[2:]:
        if not row or not row[0].strip():
            continue
        try:
            wtf_org = float(col(row, "wtf(2)"))
            log10_eta = float(col(row, "log10 viscosity (Pa.s)"))
        except ValueError:
            continue
        aw_str = col(row, "aw")
        aw = None
        if aw_str not in ("", "NaN", "nan"):
            try:
                aw = float(aw_str)
            except ValueError:
                aw = None
        T_str = col(row, "T [K]") or col(row, "T[K]")
        try:
            T_K = float(T_str) if T_str not in ("", "NaN") else T_from_name
        except ValueError:
            T_K = T_from_name
        if T_K is None or not (0 < wtf_org < 1):
            continue
        pts.append({"wtf_organic": wtf_org, "aw_measured": aw, "log10_eta_measured": log10_eta, "T_K": T_K})

    if len(pts) < 3:
        continue
    curves.append({"name": stem, "source": source, "smiles": smiles, "tg_measured_K": tg_measured_K,
                    "n_points": len(pts), "points": pts})
    n_rows_total += len(pts)

print(f"parsed {len(curves)} (compound, source) curves, {n_rows_total} total data points")
with open(OUT, "w") as f:
    json.dump(curves, f, indent=1)
print(f"wrote {OUT}")

parsed 53 (compound, source) curves, 728 total data points
wrote literature_curves.json


## 2. Tg-predictor validation and full-pipeline validation (paper Sections 3.2, 3.3)

(a) ML-predicted vs. literature-measured Tg for the 11 compounds with both.
(b) Full pipeline (ML-Tg -> VTF -> AIOMFAC -> mixture viscosity) vs. all 53 curves, plus an oracle ablation
using the measured Tg instead of the ML prediction wherever available, and a dilute-vs-glassy regime split.

In [5]:
import json
import numpy as np
from aiomfac_py.s2as import smiles_to_components
from aiomfac_py import ActivityModel, Component
from aiomfac_py.tgml_armeli import predict_tg_ml_smiles
from aiomfac_py.viscosity import organic_mixture_viscosity, pure_organic_viscosity_vtf

WATER = Component(1, "Water", ((16, 1),))

with open("literature_curves.json") as f:
    curves = json.load(f)

print("=== ML Tg predictor vs. real measured Tg ===")
tg_errs = []
smiles_seen = {}
for c in curves:
    if c["tg_measured_K"] is None or c["smiles"] in smiles_seen:
        continue
    smiles_seen[c["smiles"]] = True
    tg_ml = predict_tg_ml_smiles(c["smiles"]).tg_K
    err = tg_ml - c["tg_measured_K"]
    tg_errs.append(err)
    print(f"  {c['name']:20s} ML Tg={tg_ml:6.1f} K   measured Tg={c['tg_measured_K']:6.1f} K   error={err:+6.1f} K")
tg_errs = np.array(tg_errs)
print(f"\nMAE={np.mean(np.abs(tg_errs)):.1f} K  bias={tg_errs.mean():+.1f} K  n={len(tg_errs)}")


def organic_component(smiles):
    r = smiles_to_components([smiles], water_as_component1=True)
    return None if r.removed else r.components[1]


def predict_curve(organic, tg_K, points):
    preds = []
    for p in points:
        T_K = p["T_K"]
        try:
            eta0 = pure_organic_viscosity_vtf(T_K, tg_K)
            if not (np.isfinite(eta0) and eta0 > 0):
                preds.append(np.nan); continue
            model = ActivityModel([WATER, Component(2, "organic", organic.subgroups)])
            wtf_w = 1.0 - p["wtf_organic"]
            res = model.evaluate([wtf_w, p["wtf_organic"]], T_K, basis="mass")
            v = organic_mixture_viscosity(model, res.x, T_K, {2: eta0})
            eta = v.eta_pas
            preds.append(np.log10(eta) if (np.isfinite(eta) and eta > 0) else np.nan)
        except Exception:
            preds.append(np.nan)
    return np.array(preds)


print("\n=== pipeline (ML Tg) and oracle pipeline (measured Tg) vs. literature curves ===")
results_per_curve = []
for c in curves:
    organic = organic_component(c["smiles"])
    if organic is None:
        continue
    try:
        tg_ml = predict_tg_ml_smiles(c["smiles"]).tg_K
    except Exception:
        continue

    y_meas = np.array([p["log10_eta_measured"] for p in c["points"]])
    pred_ml = predict_curve(organic, tg_ml, c["points"])
    valid = np.isfinite(pred_ml) & np.isfinite(y_meas)
    mae_ml = float(np.mean(np.abs(pred_ml[valid] - y_meas[valid]))) if valid.any() else None

    mae_oracle = None
    if c["tg_measured_K"] is not None:
        pred_oracle = predict_curve(organic, c["tg_measured_K"], c["points"])
        valid_o = np.isfinite(pred_oracle) & np.isfinite(y_meas)
        mae_oracle = float(np.mean(np.abs(pred_oracle[valid_o] - y_meas[valid_o]))) if valid_o.any() else None

    mean_wtf = float(np.mean([p["wtf_organic"] for p in c["points"]]))
    regime = "glassy" if mean_wtf >= 0.5 else "dilute"
    results_per_curve.append({"name": c["name"], "source": c["source"], "n_points": len(c["points"]),
                               "mean_wtf_organic": mean_wtf, "regime": regime,
                               "mae_ml_tg": mae_ml, "mae_oracle_tg": mae_oracle})
    oracle_str = f"oracle-Tg MAE={mae_oracle:.2f}" if mae_oracle is not None else "oracle-Tg n/a"
    mae_str = f"{mae_ml:.2f}" if mae_ml is not None else "FAILED"
    print(f"  {c['name']:20s} {c['source']:16s} [{regime:6s}] n={len(c['points']):3d}  ML-Tg MAE={mae_str:>6s}   {oracle_str}")

valid_results = [r for r in results_per_curve if r["mae_ml_tg"] is not None]
mae_all_ml = np.array([r["mae_ml_tg"] for r in valid_results])
oracle_results = [r for r in valid_results if r["mae_oracle_tg"] is not None]
mae_oracle_subset_ml = np.array([r["mae_ml_tg"] for r in oracle_results])
mae_oracle_subset_oracle = np.array([r["mae_oracle_tg"] for r in oracle_results])

print(f"\n=== summary across {len(valid_results)} curves ===")
print(f"ALL curves, ML-predicted Tg: mean MAE = {mae_all_ml.mean():.2f} (median {np.median(mae_all_ml):.2f})")
print(f"On the {len(oracle_results)}-curve oracle subset: ML-Tg mean MAE = {mae_oracle_subset_ml.mean():.3f}  "
      f"measured-Tg mean MAE = {mae_oracle_subset_oracle.mean():.3f}")

dilute = [r["mae_ml_tg"] for r in valid_results if r["regime"] == "dilute"]
glassy = [r["mae_ml_tg"] for r in valid_results if r["regime"] == "glassy"]
print(f"\nRegime split: dilute (n={len(dilute)}) mean MAE = {np.mean(dilute):.3f}   "
      f"glassy (n={len(glassy)}) mean MAE = {np.mean(glassy):.3f}   ratio = {np.mean(glassy)/np.mean(dilute):.1f}x")

with open("validation_results.json", "w") as f:
    json.dump({"tg_predictor_mae_K": float(np.mean(np.abs(tg_errs))), "tg_predictor_bias_K": float(tg_errs.mean()),
               "n_tg_compounds": len(tg_errs), "per_curve_results": results_per_curve,
               "summary": {"all_curves_mean_mae": float(mae_all_ml.mean()), "n_all_curves": len(valid_results),
                           "oracle_subset_ml_tg_mean_mae": float(mae_oracle_subset_ml.mean()),
                           "oracle_subset_measured_tg_mean_mae": float(mae_oracle_subset_oracle.mean()),
                           "n_oracle_subset": len(oracle_results),
                           "regime_split": {"dilute_mean_mae": float(np.mean(dilute)), "n_dilute": len(dilute),
                                             "glassy_mean_mae": float(np.mean(glassy)), "n_glassy": len(glassy)}}}, f, indent=1)
print("\nwrote validation_results.json")

=== ML Tg predictor vs. real measured Tg ===


  1,2,4-butanetriol    ML Tg= 197.9 K   measured Tg= 200.7 K   error=  -2.8 K
  1,2,6-hexanetriol    ML Tg= 202.7 K   measured Tg= 204.2 K   error=  -1.5 K
  citric_acid          ML Tg= 283.2 K   measured Tg= 283.5 K   error=  -0.3 K
  erythritol           ML Tg= 226.8 K   measured Tg= 228.5 K   error=  -1.7 K
  fructose             ML Tg= 279.5 K   measured Tg= 280.8 K   error=  -1.3 K
  glucose              ML Tg= 282.3 K   measured Tg= 299.8 K   error= -17.5 K
  glycerol             ML Tg= 187.7 K   measured Tg= 186.2 K   error=  +1.5 K
  raffinose            ML Tg= 381.4 K   measured Tg= 381.7 K   error=  -0.4 K
  sorbitol             ML Tg= 273.9 K   measured Tg= 275.0 K   error=  -1.1 K
  sucrose              ML Tg= 339.1 K   measured Tg= 335.0 K   error=  +4.1 K
  trehalose            ML Tg= 380.3 K   measured Tg= 383.0 K   error=  -2.7 K

MAE=3.2 K  bias=-2.1 K  n=11

=== pipeline (ML Tg) and oracle pipeline (measured Tg) vs. literature curves ===
  1,2,4-butanetriol    Romero2

  1,2,6-hexanetriol    Song2016         [glassy] n= 11  ML-Tg MAE=  0.32   oracle-Tg MAE=0.28
  1,4-butanediol       Hawrylak1998     [glassy] n=  9  ML-Tg MAE=  0.66   oracle-Tg n/a
  1,4-butanediol       Romero2008       [dilute] n=  4  ML-Tg MAE=  0.11   oracle-Tg n/a
  1,4-butanediol       Romero2008       [dilute] n=  4  ML-Tg MAE=  0.08   oracle-Tg n/a
  1,4-butanediol       Song2016         [glassy] n= 12  ML-Tg MAE=  0.32   oracle-Tg n/a
  1,4-butanediol       Yang2004         [glassy] n= 11  ML-Tg MAE=  0.59   oracle-Tg n/a
  acetic_acid          CRCHandbook      [dilute] n= 35  ML-Tg MAE=  0.22   oracle-Tg n/a
  citric_acid          CRCHandbook      [dilute] n= 21  ML-Tg MAE=  0.06   oracle-Tg MAE=0.06
  citric_acid          Laguerie1976     [dilute] n= 10  ML-Tg MAE=  0.40   oracle-Tg MAE=0.39
  citric_acid          Song2016         [glassy] n= 15  ML-Tg MAE=  1.02   oracle-Tg MAE=1.05
  citric_acid          Song2016Bulk     [dilute] n=  5  ML-Tg MAE=  0.29   oracle-Tg MAE=0

  erythritol           Romero2008       [dilute] n=  4  ML-Tg MAE=  0.04   oracle-Tg MAE=0.04
  erythritol           Song2016         [glassy] n= 10  ML-Tg MAE=  0.73   oracle-Tg MAE=0.68
  erythritol           Zhu2010          [dilute] n= 10  ML-Tg MAE=  0.00   oracle-Tg MAE=0.00
  fructose             CRCHandbook      [dilute] n= 30  ML-Tg MAE=  0.04   oracle-Tg MAE=0.04
  fructose             Rampp2000        [glassy] n=  7  ML-Tg MAE=  0.20   oracle-Tg MAE=0.17


  fructose             Rampp2000        [glassy] n=  7  ML-Tg MAE=  0.16   oracle-Tg MAE=0.13
  fructose             Telis2007        [dilute] n=  6  ML-Tg MAE=  0.06   oracle-Tg MAE=0.05
  fructose             Telis2007        [dilute] n=  6  ML-Tg MAE=  0.06   oracle-Tg MAE=0.06
  glucose              Mazurkiewicz2001 [dilute] n=  8  ML-Tg MAE=  0.50   oracle-Tg MAE=0.41
  glucose              Song2016         [glassy] n=  9  ML-Tg MAE=  1.95   oracle-Tg MAE=1.20
  glucose              Telis2007        [dilute] n=  4  ML-Tg MAE=  0.05   oracle-Tg MAE=0.09
  glutaric_acid        Chmielewska2007  [dilute] n= 18  ML-Tg MAE=  0.06   oracle-Tg n/a
  glutaric_acid        Song2016         [dilute] n= 20  ML-Tg MAE=  0.51   oracle-Tg n/a
  glycerol             CRCHandbook      [dilute] n= 35  ML-Tg MAE=  0.11   oracle-Tg MAE=0.12
  glycerol             Mazurkiewicz2001 [dilute] n= 22  ML-Tg MAE=  0.04   oracle-Tg MAE=0.03
  glycerol             Segur1951        [glassy] n= 22  ML-Tg MAE=  0.

  glycerol             Song2016         [glassy] n=  3  ML-Tg MAE=  0.92   oracle-Tg MAE=0.99
  maleic_acid          Chmielewska2007  [dilute] n= 14  ML-Tg MAE=  0.09   oracle-Tg n/a
  maleic_acid          Gomez1986        [dilute] n=  4  ML-Tg MAE=  0.30   oracle-Tg n/a
  maleic_acid          Song2016         [dilute] n=  7  ML-Tg MAE=  1.02   oracle-Tg n/a
  maltose              Song2016         [glassy] n=  8  ML-Tg MAE=  2.13   oracle-Tg n/a
  maltose              Ueadaira1969     [dilute] n=  8  ML-Tg MAE=  0.05   oracle-Tg n/a
  raffinose            Song2016         [glassy] n=  9  ML-Tg MAE=  2.90   oracle-Tg MAE=2.90
  sorbitol             Song2016         [glassy] n=  8  ML-Tg MAE=  0.74   oracle-Tg MAE=0.74
  sorbitol             Zhu2010          [dilute] n= 10  ML-Tg MAE=  0.12   oracle-Tg MAE=0.12
  sucrose              CRCHandbook      [dilute] n= 33  ML-Tg MAE=  0.20   oracle-Tg MAE=0.20
  sucrose              Forst2002        [dilute] n=  5  ML-Tg MAE=  0.17   oracle-Tg 

  sucrose              Power2013        [glassy] n=118  ML-Tg MAE=  1.05   oracle-Tg MAE=1.12
  sucrose              Quintas2006      [glassy] n=  5  ML-Tg MAE=  1.22   oracle-Tg MAE=1.26
  sucrose              Song2016         [glassy] n=  8  ML-Tg MAE=  1.74   oracle-Tg MAE=1.64
  sucrose              Swindells1958    [dilute] n= 56  ML-Tg MAE=  0.28   oracle-Tg MAE=0.28
  sucrose              Telis2007        [dilute] n=  6  ML-Tg MAE=  0.22   oracle-Tg MAE=0.22
  trehalose            Magazu1999       [dilute] n=  9  ML-Tg MAE=  0.16   oracle-Tg MAE=0.16
  trehalose            Miller1999       [glassy] n=  4  ML-Tg MAE=  0.16   oracle-Tg MAE=0.15
  trehalose            Rampp2000        [dilute] n=  3  ML-Tg MAE=  0.15   oracle-Tg MAE=0.15
  trehalose            Song2016         [glassy] n= 15  ML-Tg MAE=  2.23   oracle-Tg MAE=2.17

=== summary across 53 curves ===
ALL curves, ML-predicted Tg: mean MAE = 0.48 (median 0.20)
On the 40-curve oracle subset: ML-Tg mean MAE = 0.480  measur

## 3. Figures: predicted-vs-measured scatter, example curves, Tg validation

In [6]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 11, "font.family": "sans-serif", "axes.spines.top": False,
                      "axes.spines.right": False, "figure.dpi": 150})
C_DILUTE, C_GLASSY, C_REF = "#2b6cb0", "#c05621", "#555555"

all_meas, all_pred, all_regime = [], [], []
example_curves = {}
for c in curves:
    organic = organic_component(c["smiles"])
    if organic is None:
        continue
    try:
        tg_ml = predict_tg_ml_smiles(c["smiles"]).tg_K
    except Exception:
        continue
    pred = predict_curve(organic, tg_ml, c["points"])
    meas = np.array([p["log10_eta_measured"] for p in c["points"]])
    mean_wtf = np.mean([p["wtf_organic"] for p in c["points"]])
    regime = "glassy" if mean_wtf >= 0.5 else "dilute"
    valid = np.isfinite(pred) & np.isfinite(meas)
    all_meas.extend(meas[valid]); all_pred.extend(pred[valid]); all_regime.extend([regime] * int(valid.sum()))
    key = f"{c['name']}_{c['source']}"
    if key in ("sucrose_CRCHandbook", "glycerol_CRCHandbook", "citric_acid_CRCHandbook", "trehalose_Song2016"):
        example_curves[key] = {"points": c["points"], "measured": meas, "predicted": pred}

all_meas, all_pred, all_regime = np.array(all_meas), np.array(all_pred), np.array(all_regime)

fig, ax = plt.subplots(figsize=(5.2, 5.0))
for regime, color, label in [("dilute", C_DILUTE, "dilute/liquid (mean $w_{org}$<0.5)"),
                               ("glassy", C_GLASSY, "concentrated/glassy (mean $w_{org}$$\\geq$0.5)")]:
    m = all_regime == regime
    ax.scatter(all_meas[m], all_pred[m], s=14, alpha=0.55, color=color, label=label, edgecolors="none")
lims = [min(all_meas.min(), all_pred.min()) - 0.5, max(all_meas.max(), all_pred.max()) + 0.5]
ax.plot(lims, lims, "--", color=C_REF, lw=1.2, zorder=0, label="1:1")
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel(r"measured $\log_{10}(\eta$ / Pa s$)$ [literature]")
ax.set_ylabel(r"pipeline-predicted $\log_{10}(\eta$ / Pa s$)$")
ax.set_title("Pipeline vs. 53 literature-measured viscosity curves\n(16 organics, 728 points)", fontsize=10.5)
ax.legend(fontsize=8.5, loc="upper left", frameon=False)
mae_all = np.mean(np.abs(all_meas - all_pred))
ax.text(0.97, 0.05, f"overall MAE = {mae_all:.2f} log$_{{10}}$ Pa s", transform=ax.transAxes,
        ha="right", fontsize=8.5, color="#333333")
fig.tight_layout()
fig.savefig("fig_A_predicted_vs_measured.png", dpi=220)
plt.show()
print("wrote fig_A_predicted_vs_measured.png")

wrote fig_A_predicted_vs_measured.png


/var/folders/4x/8kn2nym12cn7x7qmg_6s4b8h0000gn/T/ipykernel_55132/1634548537.py:48: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
fig, axes = plt.subplots(2, 2, figsize=(8.5, 7.0))
titles = {"sucrose_CRCHandbook": "Sucrose (CRC Handbook, 293 K)",
          "glycerol_CRCHandbook": "Glycerol (CRC Handbook, 293 K)",
          "citric_acid_CRCHandbook": "Citric acid (CRC Handbook, 293 K)",
          "trehalose_Song2016": "Trehalose (Song et al. 2016, 293 K) -- glassy regime"}
for ax, (key, d) in zip(axes.flat, example_curves.items()):
    wtf = np.array([p["wtf_organic"] for p in d["points"]])
    order = np.argsort(wtf)
    ax.plot(wtf[order], d["predicted"][order], "-", color=C_GLASSY if "trehalose" in key else C_DILUTE,
            lw=2, label="pipeline (ML Tg)")
    ax.scatter(wtf, d["measured"], s=22, color="black", zorder=5, label="literature measurement")
    ax.set_title(titles[key], fontsize=10)
    ax.set_xlabel("organic mass fraction")
    ax.set_ylabel(r"$\log_{10}(\eta$ / Pa s$)$")
    ax.legend(fontsize=8, frameon=False, loc="upper left")
fig.suptitle("Pipeline-predicted vs. measured viscosity curves: representative examples", fontsize=12)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig("fig_B_example_curves.png", dpi=220)
plt.show()
print("wrote fig_B_example_curves.png")

wrote fig_B_example_curves.png


/var/folders/4x/8kn2nym12cn7x7qmg_6s4b8h0000gn/T/ipykernel_55132/2301843130.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
tg_pairs = []
smiles_seen2 = set()
for c in curves:
    if c["tg_measured_K"] is None or c["smiles"] in smiles_seen2:
        continue
    smiles_seen2.add(c["smiles"])
    try:
        tg_ml = predict_tg_ml_smiles(c["smiles"]).tg_K
    except Exception:
        continue
    tg_pairs.append((c["name"], c["tg_measured_K"], tg_ml))

fig, ax = plt.subplots(figsize=(4.6, 4.6))
meas_tg = np.array([p[1] for p in tg_pairs])
ml_tg = np.array([p[2] for p in tg_pairs])
ax.scatter(meas_tg, ml_tg, s=50, color=C_DILUTE, zorder=5)
offsets = {"raffinose": (6, 10), "trehalose": (6, -8), "sucrose": (6, -4),
           "citric acid": (6, 6), "fructose": (-55, -2), "sorbitol": (-58, -8)}
for name, mt, pt in tg_pairs:
    dx, dy = offsets.get(name, (5, 5))
    ax.annotate(name.replace("_", " "), (mt, pt), fontsize=7, xytext=(dx, dy), textcoords="offset points")
lims = [meas_tg.min() - 10, meas_tg.max() + 10]
ax.plot(lims, lims, "--", color=C_REF, lw=1.2)
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel("measured $T_g$ [K] (BIMOG database)")
ax.set_ylabel("ML-predicted $T_g$ [K] (Armeli et al. 2023 model)")
mae_tg = np.mean(np.abs(meas_tg - ml_tg))
ax.set_title(f"Glass-transition-temperature predictor vs. real data\nMAE = {mae_tg:.1f} K  (n={len(tg_pairs)})",
             fontsize=10.5)
fig.tight_layout()
fig.savefig("fig_C_tg_validation.png", dpi=220)
plt.show()
print("wrote fig_C_tg_validation.png")

wrote fig_C_tg_validation.png


/var/folders/4x/8kn2nym12cn7x7qmg_6s4b8h0000gn/T/ipykernel_55132/2168896745.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Glassy-regime diagnosis (paper Section 3.4.1): pure-component viscosity vs. the mixing rule

Holds the AIOMFAC/G2020 mixing rule and composition fixed per curve and searches for the pure-component
viscosity (eta0) that would have minimized that curve's error, then compares it to the eta0 the pipeline
actually used (from ML-Tg -> VTF).

In [9]:
from scipy.optimize import minimize_scalar

def mixture_logeta_given_eta0(model, T_K, points, log10_eta0):
    eta0 = 10.0 ** log10_eta0
    preds = []
    for pt in points:
        wtf_w = 1.0 - pt["wtf_organic"]
        res = model.evaluate([wtf_w, pt["wtf_organic"]], pt["T_K"], basis="mass")
        v = organic_mixture_viscosity(model, res.x, pt["T_K"], {2: eta0})
        eta = v.eta_pas
        preds.append(np.log10(eta) if (np.isfinite(eta) and eta > 0) else np.nan)
    return np.array(preds)


diag_results = []
for c in curves:
    organic = organic_component(c["smiles"])
    if organic is None:
        continue
    model = ActivityModel([WATER, Component(2, "organic", organic.subgroups)])
    y_meas = np.array([pt["log10_eta_measured"] for pt in c["points"]])
    mean_wtf = np.mean([pt["wtf_organic"] for pt in c["points"]])
    regime = "glassy" if mean_wtf >= 0.5 else "dilute"

    try:
        tg_ml = predict_tg_ml_smiles(c["smiles"]).tg_K
        eta0_pipeline = pure_organic_viscosity_vtf(c["points"][0]["T_K"], tg_ml)
        pred_pipeline = mixture_logeta_given_eta0(model, c["points"][0]["T_K"], c["points"], np.log10(eta0_pipeline))
    except Exception:
        continue
    valid = np.isfinite(pred_pipeline) & np.isfinite(y_meas)
    if valid.sum() < 3:
        continue
    mae_pipeline = float(np.mean(np.abs(pred_pipeline[valid] - y_meas[valid])))

    def obj(log10_eta0):
        pred = mixture_logeta_given_eta0(model, c["points"][0]["T_K"], c["points"], log10_eta0)
        v = np.isfinite(pred) & np.isfinite(y_meas)
        return float(np.mean(np.abs(pred[v] - y_meas[v]))) if v.sum() else 1e6

    res = minimize_scalar(obj, bounds=(-6, 20), method="bounded", options={"xatol": 0.01})
    diag_results.append({"name": c["name"], "source": c["source"], "regime": regime, "n": int(valid.sum()),
                          "log10_eta0_pipeline": float(np.log10(eta0_pipeline)),
                          "log10_eta0_optimal": float(res.x),
                          "mae_pipeline": mae_pipeline, "mae_optimal_eta0": res.fun})

by_regime = {"dilute": [], "glassy": []}
for r in diag_results:
    by_regime[r["regime"]].append(r)
print(f"{'name':20s} {'source':16s} {'regime':7s} {'eta0_pipe':>10s} {'eta0_opt':>9s} {'MAE_pipe':>9s} {'MAE_opt':>8s}")
for r in sorted(diag_results, key=lambda r: -r["mae_pipeline"])[:20]:
    print(f"{r['name']:20s} {r['source']:16s} {r['regime']:7s} {r['log10_eta0_pipeline']:10.2f} "
          f"{r['log10_eta0_optimal']:9.2f} {r['mae_pipeline']:9.2f} {r['mae_optimal_eta0']:8.2f}")
print("\n=== summary by regime ===")
for regime, rs in by_regime.items():
    print(f"{regime:8s} n={len(rs):3d}  mean MAE (pipeline eta0) = {np.mean([r['mae_pipeline'] for r in rs]):.3f}   "
          f"mean MAE (optimal eta0) = {np.mean([r['mae_optimal_eta0'] for r in rs]):.3f}")

with open("glassy_regime_diagnosis.json", "w") as f:
    json.dump(diag_results, f, indent=1)
print("\nwrote glassy_regime_diagnosis.json")

name                 source           regime   eta0_pipe  eta0_opt  MAE_pipe  MAE_opt
raffinose            Song2016         glassy       31.50     20.00      2.90     3.01
trehalose            Song2016         glassy       31.11     20.00      2.23     2.83
maltose              Song2016         glassy       26.28     20.00      2.13     2.38
glucose              Song2016         glassy        9.33     17.47      1.95     1.04
sucrose              Song2016         glassy       19.75     12.80      1.74     1.21
sucrose              Quintas2006      glassy       19.72     20.00      1.22     1.21
sucrose              Power2013        glassy       19.75     20.00      1.05     1.03
citric_acid          Song2016         glassy        9.49      7.57      1.02     0.43
maleic_acid          Song2016         dilute        4.42     -0.85      1.02     0.03
glycerol             Song2016         glassy       -0.48      0.31      0.92     0.27
sorbitol             Song2016         glassy        7.

## 5. Strategy tests (paper Sections 3.4.2, 3.4.3): fragility-parameter sensitivity, and an ideal-mixing-rule
ablation on the deep-glassy sugars

**(A)** For curves where eta0-tuning closes most of the gap: could a physically plausible fragility parameter D
(G2020's own 5-30 range), at the UNCHANGED ML-predicted Tg, reproduce the optimal eta0 found above?

**(B)** For the curves where NO eta0 closes the gap (the deepest-glassy sugars): does G2020's non-ideal
group-residual mixing correction itself hurt, relative to the simplest possible ideal (log-linear) mixing rule?

*Note:* the originally considered alternative -- implementing LZ2022's third, unimplemented "ZSR-style" mixing
rule -- turned out not to apply here: that rule only combines an electrolyte sub-model with the organic one for
mixtures that also contain inorganic ions. These curves are ion-free water-organic binaries, for which
`organic_mixture_viscosity` already *is* the complete G2020 model, so Test (B) below (the ideal-mixing
ablation) is the applicable way to ask the same underlying question.

In [10]:
from scipy.optimize import brentq
from aiomfac_py.viscosity import water_viscosity_pas

diag_by_key = {(d["name"], d["source"]): d for d in diag_results}

print("=== Test A: fragility parameter (D) required to reproduce the optimal eta0, at fixed ML Tg ===")
EXPLAINABLE = [("glycerol", "Segur1951"), ("glycerol", "Song2016"), ("citric_acid", "Song2016"),
               ("erythritol", "Song2016"), ("1,4-butanediol", "Hawrylak1998"), ("1,4-butanediol", "Yang2004"),
               ("glucose", "Song2016"), ("sorbitol", "Song2016")]
for name, source in EXPLAINABLE:
    key = (name, source)
    if key not in diag_by_key:
        continue
    d = diag_by_key[key]
    c = next((c for c in curves if c["name"] == name and c["source"] == source), None)
    T_K = c["points"][0]["T_K"]
    tg_ml = predict_tg_ml_smiles(c["smiles"]).tg_K
    target = d["log10_eta0_optimal"]

    def f(D):
        T0 = 39.17 * tg_ml / (D + 39.17)
        if T_K <= T0:
            return 1e6
        return (-5.0 + 0.434 * T0 * D / (T_K - T0)) - target

    try:
        D_opt = brentq(f, 0.5, 300.0, xtol=0.01)
        verdict = "PLAUSIBLE" if D_opt <= 50 else "IMPLAUSIBLE"
        D_str = f"{D_opt:.1f}"
    except ValueError:
        D_str, verdict = "n/a", "NO SOLUTION -- Tg itself would need to change"
    print(f"  {name:16s} {source:14s} Tg_ML={tg_ml:6.1f}K  D_required={D_str:>6s}   {verdict}")

print("\n=== Test B: ideal (no group-residual) mixing rule vs. G2020 full model, deep-glassy sugars ===")
UNEXPLAINABLE = [("raffinose", "Song2016"), ("trehalose", "Song2016"), ("maltose", "Song2016")]
for name, source in UNEXPLAINABLE:
    key = (name, source)
    d = diag_by_key.get(key)
    c = next((c for c in curves if c["name"] == name and c["source"] == source), None)
    if d is None or c is None:
        continue
    organic = organic_component(c["smiles"])
    model = ActivityModel([WATER, Component(2, "organic", organic.subgroups)])
    y_meas = np.array([pt["log10_eta_measured"] for pt in c["points"]])
    T_K = c["points"][0]["T_K"]

    for label, log10_eta0 in [("pipeline eta0", d["log10_eta0_pipeline"]), ("optimal eta0", d["log10_eta0_optimal"])]:
        eta0_organic = 10.0 ** log10_eta0
        preds_g2020, preds_ideal = [], []
        for pt in c["points"]:
            wtf_w = 1.0 - pt["wtf_organic"]
            res = model.evaluate([wtf_w, pt["wtf_organic"]], T_K, basis="mass")
            eta0_w = water_viscosity_pas(T_K)
            v = organic_mixture_viscosity(model, res.x, T_K, {2: eta0_organic})
            preds_g2020.append(v.log10_eta_pas if np.isfinite(v.eta_pas) and v.eta_pas > 0 else np.nan)
            preds_ideal.append(float(np.dot(res.x, np.log([eta0_w, eta0_organic]))) / 2.302585093)
        preds_g2020, preds_ideal = np.array(preds_g2020), np.array(preds_ideal)
        v1, v2 = np.isfinite(preds_g2020), np.isfinite(preds_ideal)
        mae_g2020 = float(np.mean(np.abs(preds_g2020[v1] - y_meas[v1]))) if v1.any() else None
        mae_ideal = float(np.mean(np.abs(preds_ideal[v2] - y_meas[v2]))) if v2.any() else None
        print(f"  {name:12s} {source:10s} [{label:14s}]  G2020(full) MAE={mae_g2020:.2f}   ideal-log-linear MAE={mae_ideal:.2f}")

=== Test A: fragility parameter (D) required to reproduce the optimal eta0, at fixed ML Tg ===
  glycerol         Segur1951      Tg_ML= 187.7K  D_required=  12.8   PLAUSIBLE
  glycerol         Song2016       Tg_ML= 187.7K  D_required=  13.4   PLAUSIBLE
  citric_acid      Song2016       Tg_ML= 283.2K  D_required=   4.3   PLAUSIBLE
  erythritol       Song2016       Tg_ML= 226.8K  D_required=  19.9   PLAUSIBLE
  1,4-butanediol   Hawrylak1998   Tg_ML= 158.0K  D_required=  16.0   PLAUSIBLE
  1,4-butanediol   Yang2004       Tg_ML= 158.0K  D_required=  17.2   PLAUSIBLE
  glucose          Song2016       Tg_ML= 282.3K  D_required=   n/a   NO SOLUTION -- Tg itself would need to change
  sorbitol         Song2016       Tg_ML= 273.9K  D_required=  86.6   IMPLAUSIBLE

=== Test B: ideal (no group-residual) mixing rule vs. G2020 full model, deep-glassy sugars ===
  raffinose    Song2016   [pipeline eta0 ]  G2020(full) MAE=2.90   ideal-log-linear MAE=2.02
  raffinose    Song2016   [optimal eta0  ]  G2

## 6. Download results

In [11]:
import shutil
shutil.make_archive("validation_part1_results", "zip", ".", logger=None) if False else None
import glob as _glob
outputs = ["literature_curves.json", "validation_results.json", "glassy_regime_diagnosis.json",
           "fig_A_predicted_vs_measured.png", "fig_B_example_curves.png", "fig_C_tg_validation.png"]
try:
    from google.colab import files
    for fn in outputs:
        if os.path.exists(fn):
            files.download(fn)
except ImportError:
    print("Not running in Colab -- output files are in the current directory:", outputs)

Not running in Colab -- output files are in the current directory: ['literature_curves.json', 'validation_results.json', 'glassy_regime_diagnosis.json', 'fig_A_predicted_vs_measured.png', 'fig_B_example_curves.png', 'fig_C_tg_validation.png']
